## 1) Links Collection

In [1]:
import requests
from tqdm import tqdm
import time
import random
from fake_useragent import UserAgent

In [2]:
headers = {'User-Agent': 'VacancyCollector (nikischov.dan@yandex.ru)'}
vacancies_url = 'https://api.hh.ru/vacancies?host=hh.ru'
vacancy_ids = []
relevant_proles = [84, 116, 36, 157, 125, 156, 160, 10, 150, 25, 165, 34, 73, 
                   155, 96, 164, 104, 107, 112, 113, 148, 114, 121, 124, 126]

In [3]:
len(sorted([84, 116, 36, 157, 125, 156, 160, 10, 150, 25, 165, 34, 73, 
                   155, 96, 164, 104, 107, 112, 113, 148, 114, 121, 124, 126]))

25

In [ ]:
for december_day in tqdm(range(1, 32)):

    day = str(december_day)
    if len(day) == 1:
        day = '0' + day
    date = '2025-12-' + day
    
    for prole in relevant_proles:
        for page in range(3):
            parameters = {
                'page': str(page),
                'per_page': '100',
                'professional_role': str(prole),
                "date_from": date,
                "date_to": date
                }
            vacancies = requests.get(url = vacancies_url, params = parameters, headers = headers)
            json_v = vacancies.json()
            if page == 0:
                print(f'{date}, профессиональная роль {prole}. Найдено вакансий {json_v['found']}')
            for j in range(100):
                try:
                    vacancy_ids.append(json_v['items'][j]['id'])
                except:
                    continue
            time.sleep(random.uniform(0.5, 1))

## 2) Missed Vacancies Loading


1) 2025-12-22, professional role 96. Vacancies found: 378

In [20]:
for page in range(3, 4):
    parameters = {
        'page': str(page),
        'per_page': '100',
        'professional_role': '96',
        "date_from": '2025-12-22',
        "date_to": '2025-12-22'
        }
    vacancies = requests.get(url = vacancies_url, params = parameters, headers = headers)
    json_v = vacancies.json()
    for j in range(100):
        try:
            vacancy_ids.append(json_v['items'][j]['id'])
        except:
            continue
    time.sleep(random.uniform(0.5, 1))

2) 2025-12-25, professional role 121. Vacancies found: 596

In [21]:
for page in range(3, 6):
    parameters = {
        'page': str(page),
        'per_page': '100',
        'professional_role': '121',
        "date_from": '2025-12-25',
        "date_to": '2025-12-25'
        }
    vacancies = requests.get(url = vacancies_url, params = parameters, headers = headers)
    json_v = vacancies.json()
    for j in range(100):
        try:
            vacancy_ids.append(json_v['items'][j]['id'])
        except:
            continue
    time.sleep(random.uniform(0.5, 1))

3) 2025-12-26, professional role 121. Vacancies found: 370

In [22]:
for page in range(3, 4):
    parameters = {
        'page': str(page),
        'per_page': '100',
        'professional_role': '121',
        "date_from": '2025-12-26',
        "date_to": '2025-12-26'
        }
    vacancies = requests.get(url = vacancies_url, params = parameters, headers = headers)
    json_v = vacancies.json()
    for j in range(100):
        try:
            vacancy_ids.append(json_v['items'][j]['id'])
        except:
            continue
    time.sleep(random.uniform(0.5, 1))

In [23]:
len(vacancy_ids)

12359

In [24]:
with open('vacancies.txt', 'x', encoding='utf-8') as file:
    for id in set(vacancy_ids):
        file.write(id + '\n')

## 3) Vacancies Parsing

In [25]:
def delete_tags(text):
    tags = ['<p>', '</p>', 
            '<strong>', '</strong>', 
            '<ul>', '</ul>', 
            '<li>', '</li>',
            '<ol>', '</ol>',
            '<em>', '</em>',
            '<div>', '</div>',
            '<h1>', '</h1>', '<h2>', '</h2>',
            '<h3>', '</h3>', '<h4>', '</h4>',
            '<h5>', '</h5>', '<h6>', '</h6>',
            '<span>', '</span>',
            '<br />', '&quot', '&amp', '&gt']
    for tag in tags:
        text = text.replace(tag, ' ')
    return text

In [ ]:
import pandas as pd

In [4]:
vacancies_df = pd.DataFrame(columns=['Id', 'Name', 'Description', 'Professional_Role_Id', 'Professional_Role'])
headers = {'User-Agent': 'VacancyCollector (nikischov.dan@yandex.ru)'}
basic_url = 'https://api.hh.ru/vacancies/{:d}'
relevant_proles = [84, 116, 36, 157, 125, 156, 160, 10, 150, 25, 165, 34, 73, 
                   155, 96, 164, 104, 107, 112, 113, 148, 114, 121, 124, 126]
# irrelevant_vacancies = 0
vacancies_with_mistakes = 0
good_vacancies = 0

In [5]:
with open('vacancies.txt', 'r', encoding='utf-8') as file:
    n_vacancy = 1
    trial = 1
    for id in file:
        id = int(id.strip())
        try:
            new_vacancy = requests.get(url = basic_url.format(id), headers=headers)
            new_json = new_vacancy.json()
            p_role_id = int(new_json['professional_roles'][0]['id'])
            # if p_role_id in relevant_proles:
            name = new_json['name']
            description = new_json['description']
            p_role = new_json['professional_roles'][0]['name']
            vacancies_df.loc[n_vacancy] = [id, name, description, p_role_id, p_role]
            print(f'Вакансия {trial} {name} успешно добавлена ✅')
            good_vacancies += 1
            n_vacancy += 1
            trial += 1
            # if trial == 500:
            #     break
            time.sleep(random.uniform(0.5, 1))
            # else:
            #     print(f'Вакансия {trial} {new_json['name']} не добавлена - нерелевантная профессиональная роль ❌')
            #     irrelevant_vacancies += 1
            #     trial += 1
            #     if trial == 500:
            #         break
            #     time.sleep(random.uniform(2, 3))
            #     continue
        except:
            print(f'С вакансией {trial} что-то пошло не так ❌')
            vacancies_with_mistakes += 1
            trial += 1
            # if trial == 500:
            #     break
            time.sleep(random.uniform(0.5, 1))
            continue
vacancies_df.to_csv('vacancies_with_tags.csv', index=False)
print(f'''
Успешно добавлено {good_vacancies} вакансий
При добалении {vacancies_with_mistakes} вакансий возникли ошибки
'''
)

Вакансия 1 Специалист технической поддержки успешно добавлена ✅
Вакансия 2 3D-визуализатор успешно добавлена ✅
Вакансия 3 Главный разработчик Python успешно добавлена ✅
Вакансия 4 GenAI Product Analyst успешно добавлена ✅
Вакансия 5 Системный аналитик (ML) успешно добавлена ✅
Вакансия 6 Специалист по операционной эффективности успешно добавлена ✅
Вакансия 7 Программист 1С успешно добавлена ✅
Вакансия 8 Программист 1С-Битрикс успешно добавлена ✅
С вакансией 9 что-то пошло не так ❌
Вакансия 10 Специалист технической поддержки успешно добавлена ✅
Вакансия 11 С# Developer успешно добавлена ✅
Вакансия 12 Руководитель группы разработки/Технический лидер в Big Data (TechLead) успешно добавлена ✅
Вакансия 13 Senior Unity Developer (Tech Dev) успешно добавлена ✅
Вакансия 14 Ведущий системный аналитик 1C ERP успешно добавлена ✅
Вакансия 15 Python Backend-разработчик (ОФИС) успешно добавлена ✅
С вакансией 16 что-то пошло не так ❌
Вакансия 17 Системный администратор успешно добавлена ✅
Вакансия 18